# Divergencia léxica y detección temprana de cambios de agenda

Este notebook es una **Etapa 2.5**: un puente entre la exploración léxica de
`03-eda-agenda.ipynb` y el modelado de tópicos (Etapa 4). Responde una versión
simplificada de O2 (`CLAUDE.md`) **sin** comprometerse todavía a un número de tópicos K:

> ¿Cuándo cambia significativamente la distribución de vocabulario de la prensa
> colombiana, usando la distribución de términos cruda en vez de tópicos latentes?

Se usan tres señales independientes, cada una con una justificación distinta (sección
14 y 24 de `CLAUDE.md` piden justificar cada técnica, no aplicarlas porque sí):

1. **Jensen-Shannon divergence** sobre la distribución de frecuencia de términos por
   período — compara distribuciones de probabilidad, tal como sugiere la sección 14.
2. **Cosine distance** sobre vectores TF-IDF agregados por período — señal
   complementaria mencionada en la misma sección, con una ponderación distinta
   (penaliza menos los términos muy frecuentes).
3. **Burst detection (Kleinberg, 2002)**, versión de 2 estados — detecta, término por
   término, cuándo su frecuencia se dispara por encima de su tasa base. No requiere
   comparar distribuciones completas; es una técnica clásica de minería de flujos de
   texto/IR, y sirve de validación cruzada: si coincide con los picos de JS/cosine, es
   evidencia convergente.

Si las tres señales coinciden en los mismos períodos, hay buena razón para creer que el
cambio es real y no un artefacto de una métrica en particular — antes de invertir en
LDA/NMF.

## Resolución temporal: adaptativa, justificada por el EDA

`03-eda-agenda.ipynb` mostró que 1980–1989 tiene muy pocos documentos y proviene casi
exclusivamente de *Semana* (revista, no diario) — no eliminamos esos años, pero tampoco
fingimos que tienen la misma resolución que el resto:

- **1980–1989**: bins de **bienio** (5 bins de 2 años) — agrupa la escasez en vez de
  esconderla en años con 1-2 documentos.
- **1990–2011**: bins de **trimestre** — el EDA mostró solo 4% de trimestres vacíos
  desde 1990, volumen suficiente para esa resolución.

Cada punto de las curvas de divergencia se reporta junto a su `N` (documentos en ese
período) y a si un período está dominado por una sola fuente, para poder distinguir una
señal real de ruido por escasez de datos — no para ocultar la limitación, sino para
que quede visible en el propio gráfico.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances

from utils import load_corpus, normalizar_texto, results_dir, setup

setup()

df = load_corpus()
RESULTS = results_dir()
print(df.shape)

## 1. Asignación de período (resolución adaptativa)

In [ ]:
def periodo_adaptativo(fecha: pd.Timestamp) -> str:
    """Bienio para 1980-1989 (escasez de datos), trimestre desde 1990."""
    anio = fecha.year
    if anio < 1990:
        inicio = 1980 + ((anio - 1980) // 2) * 2
        return f"{inicio}-{inicio + 1}"
    trimestre = (fecha.month - 1) // 3 + 1
    return f"{anio}Q{trimestre}"


def clave_orden(periodo: str):
    """Clave de ordenamiento cronológico (no alfabético) para mezclar bienios y trimestres."""
    if "Q" in periodo:
        anio_str, trimestre_str = periodo.split("Q")
        return (int(anio_str), int(trimestre_str))
    return (int(periodo.split("-")[0]), 0)


df["periodo"] = df["fecha"].apply(periodo_adaptativo)
periodos_orden = sorted(df["periodo"].unique(), key=clave_orden)

N_por_periodo = df["periodo"].value_counts().reindex(periodos_orden)

# fuente dominante por período (para anotar si un pico coincide con un período
# cubierto casi exclusivamente por un solo medio, ver sección 17-18 de CLAUDE.md)
pct_fuente_dominante = (
    df.groupby("periodo", observed=True)["fuente"]
    .apply(lambda s: s.value_counts(normalize=True).iloc[0])
    .reindex(periodos_orden)
)

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(range(len(periodos_orden)), N_por_periodo.values, color="#444")
ax.set_xticks(range(0, len(periodos_orden), 4))
ax.set_xticklabels([periodos_orden[i] for i in range(0, len(periodos_orden), 4)], rotation=90)
ax.set_ylabel("documentos")
ax.set_title("Documentos por período (resolución adaptativa: bienio 1980-89, trimestre 1990-2011)")
plt.tight_layout()
plt.show()

print(f"{len(periodos_orden)} períodos totales")
print(f"períodos con N < 30 documentos: {(N_por_periodo < 30).sum()}")
print(f"períodos con >90% de documentos de una sola fuente: {(pct_fuente_dominante > 0.9).sum()}")

## 2. Preprocesamiento léxico

Misma tokenización de `02-textometria.ipynb`/`03-eda-agenda.ipynb` (spaCy en blanco,
minúsculas, sin puntuación/espacios), agregando remoción de stopwords (NLTK español,
igual que el resto del proyecto) porque aquí sí interesa la distribución de contenido,
no la frecuencia bruta de función gramatical.

In [ ]:
import spacy
from nltk.corpus import stopwords

sw = set(stopwords.words("spanish"))

df["texto_rep"] = normalizar_texto(df["texto"])
nlp = spacy.blank("es")

tokens_por_doc = []
for doc in nlp.pipe(df["texto_rep"], batch_size=200):
    tokens_por_doc.append(
        [t.lower_ for t in doc if not t.is_punct and not t.is_space and t.lower_ not in sw]
    )
df["tokens"] = tokens_por_doc

print(f"tokens (sin stopwords): {sum(len(t) for t in tokens_por_doc):,}")

## 3. Distribución de términos por período y Jensen-Shannon divergence

`03-eda-agenda.ipynb` mostró que el vocabulario tiene una cola larguísima (305,904
tipos, curva de crecimiento sin aplanarse). Para la distribución de probabilidad por
período restringimos a los **N_TOP términos más frecuentes del corpus completo**: el
resto de la cola aporta masa de probabilidad despreciable por período y solo añade
ruido/dimensionalidad a la comparación entre distribuciones — es la misma
justificación de un `min_df`, aplicada aquí como top-K global.

In [ ]:
N_TOP = 5000

freq_global = Counter(tok for toks in df["tokens"] for tok in toks)
vocab_top = [w for w, _ in freq_global.most_common(N_TOP)]
vocab_idx = {w: i for i, w in enumerate(vocab_top)}

conteos = np.zeros((len(periodos_orden), N_TOP))
periodo_idx = {p: i for i, p in enumerate(periodos_orden)}

for periodo, toks in zip(df["periodo"], df["tokens"]):
    fila = periodo_idx[periodo]
    for tok in toks:
        j = vocab_idx.get(tok)
        if j is not None:
            conteos[fila, j] += 1

total_por_periodo = conteos.sum(axis=1)
dist = conteos / total_por_periodo[:, None]

print(f"cobertura del top-{N_TOP} sobre el total de tokens: {conteos.sum() / sum(len(t) for t in df['tokens']):.1%}")

In [ ]:
js_divergence = [np.nan]
for i in range(1, len(periodos_orden)):
    js_divergence.append(jensenshannon(dist[i], dist[i - 1], base=2))
js_divergence = pd.Series(js_divergence, index=periodos_orden, name="js_divergence")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(len(periodos_orden)), js_divergence.values, color="#444", marker="o", ms=3)

# marcar períodos de baja confianza (N chico o una sola fuente dominante)
baja_confianza = (N_por_periodo < 30) | (pct_fuente_dominante > 0.9)
for i, low in enumerate(baja_confianza.values):
    if low:
        ax.axvspan(i - 0.5, i + 0.5, color="crimson", alpha=0.12)

ax.set_xticks(range(0, len(periodos_orden), 4))
ax.set_xticklabels([periodos_orden[i] for i in range(0, len(periodos_orden), 4)], rotation=90)
ax.set_ylabel("JS divergence (base 2)")
ax.set_title("Divergencia de la distribución de términos vs. período anterior\n(sombreado = N<30 o >90% de una sola fuente: interpretar con cautela)")
plt.tight_layout()
plt.show()

## 4. TF-IDF + cosine distance (señal complementaria)

Mismo corpus, representación distinta: TF-IDF pondera hacia abajo los términos que
aparecen en casi todos los documentos (más agresivo que solo quitar stopwords) y hacia
arriba los distintivos de pocos documentos. Si el pico de JS divergence de la sección 3
se repite aquí, es una señal más robusta de cambio real.

In [ ]:
vectorizador = TfidfVectorizer(analyzer=lambda toks: toks, min_df=5, max_df=0.7)
X = vectorizador.fit_transform(df["tokens"])

n_periodos = len(periodos_orden)
X_periodo = np.zeros((n_periodos, X.shape[1]))
conteo_docs = np.zeros(n_periodos)

filas_periodo = df["periodo"].map(periodo_idx).to_numpy()
X_csr = X.tocsr()
for i in range(len(df)):
    p = filas_periodo[i]
    X_periodo[p] += X_csr[i].toarray().ravel()
    conteo_docs[p] += 1

X_periodo = X_periodo / conteo_docs[:, None]

cosine_dist = [np.nan]
for i in range(1, n_periodos):
    d = cosine_distances(X_periodo[i : i + 1], X_periodo[i - 1 : i])[0, 0]
    cosine_dist.append(d)
cosine_dist = pd.Series(cosine_dist, index=periodos_orden, name="cosine_distance")

print(f"vocabulario TF-IDF (min_df=5, max_df=0.7): {X.shape[1]:,} términos")

**Nota de rendimiento**: el bucle anterior densifica una fila a la vez (`.toarray()`
por documento) en vez de toda la matriz junta, para no materializar una matriz densa de
91,585 × vocabulario en memoria. Si el vocabulario TF-IDF resulta muy grande en tu
ejecución, esto puede tardar unos minutos en Colab; es un costo de una sola vez.

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 4))
ax1.plot(range(n_periodos), js_divergence.values, color="#444", label="JS divergence")
ax1.set_ylabel("JS divergence", color="#444")

ax2 = ax1.twinx()
ax2.plot(range(n_periodos), cosine_dist.values, color="crimson", label="cosine distance", alpha=0.8)
ax2.set_ylabel("cosine distance", color="crimson")

ax1.set_xticks(range(0, n_periodos, 4))
ax1.set_xticklabels([periodos_orden[i] for i in range(0, n_periodos, 4)], rotation=90)
ax1.set_title("JS divergence vs. cosine distance — ¿coinciden los picos?")
fig.tight_layout()
plt.show()

## 5. Burst detection (Kleinberg, 2002 — versión de 2 estados)

Implementación de la variante simplificada de 2 estados (la misma que usa por defecto
el paquete `bursts` de R): para cada término, compara su tasa base de uso contra una
tasa "de ráfaga" (`s` veces mayor) usando un autómata de costos mínimos — entrar en
ráfaga cuesta `gamma * log(n_períodos)` por nivel, salir no cuesta nada. Se aplica a
los términos más frecuentes del corpus (no a los 5000, sería redundante con las
secciones anteriores); interesa ver *cuántos* términos entran en ráfaga en cada
período, como una tercera señal independiente.

In [ ]:
def kleinberg_2estados(d, r, s: float = 2.0, gamma: float = 1.0) -> np.ndarray:
    """Detección de ráfagas de Kleinberg (2002), versión simplificada de 2 estados.

    d: conteos del término por período. r: total de tokens del período (denominador).
    s: cuánto más alta es la tasa en estado de ráfaga respecto a la tasa base.
    gamma: costo de entrar en ráfaga (mayor gamma = más evidencia requerida).
    Devuelve un array 0/1 por período (1 = en ráfaga).
    """
    d = np.asarray(d, dtype=float)
    r = np.asarray(r, dtype=float)
    n = len(d)

    p0 = min(max(d.sum() / r.sum(), 1e-10), 1 - 1e-10)
    p1 = min(p0 * s, 1 - 1e-10)

    def neg_log_lik(p, di, ri):
        p = min(max(p, 1e-10), 1 - 1e-10)
        return -(di * np.log(p) + (ri - di) * np.log(1 - p))

    costo = np.zeros((2, n))
    for t in range(n):
        costo[0, t] = neg_log_lik(p0, d[t], r[t])
        costo[1, t] = neg_log_lik(p1, d[t], r[t])

    def tau(i, j):
        return gamma * (j - i) * np.log(n) if j > i else 0.0

    C = np.zeros((2, n))
    back = np.zeros((2, n), dtype=int)
    C[:, 0] = costo[:, 0]

    for t in range(1, n):
        for i in (0, 1):
            opciones = [C[j, t - 1] + tau(j, i) for j in (0, 1)]
            j_opt = int(np.argmin(opciones))
            back[i, t] = j_opt
            C[i, t] = costo[i, t] + opciones[j_opt]

    estado_final = int(np.argmin(C[:, -1]))
    estados = np.zeros(n, dtype=int)
    estados[-1] = estado_final
    for t in range(n - 1, 0, -1):
        estados[t - 1] = back[estados[t], t]
    return estados

In [ ]:
N_TERMINOS_BURST = 150
terminos_burst = vocab_top[:N_TERMINOS_BURST]

matriz_burst = np.zeros((n_periodos, N_TERMINOS_BURST), dtype=int)
for j, termino in enumerate(terminos_burst):
    d_termino = conteos[:, vocab_idx[termino]]
    matriz_burst[:, j] = kleinberg_2estados(d_termino, total_por_periodo, s=2.0, gamma=1.0)

n_rafagas_por_periodo = pd.Series(matriz_burst.sum(axis=1), index=periodos_orden, name="n_terminos_en_rafaga")

fig, ax = plt.subplots(figsize=(13, 4))
ax.bar(range(n_periodos), n_rafagas_por_periodo.values, color="#444")
ax.set_xticks(range(0, n_periodos, 4))
ax.set_xticklabels([periodos_orden[i] for i in range(0, n_periodos, 4)], rotation=90)
ax.set_ylabel(f"# de los {N_TERMINOS_BURST} términos más frecuentes en ráfaga")
ax.set_title("Cantidad de términos en ráfaga (Kleinberg) por período")
plt.tight_layout()
plt.show()

## 6. Comparación de las tres señales

Normalizamos cada señal a [0, 1] (min-max) solo para poder verlas juntas en el mismo
eje — la escala absoluta de cada una no es comparable, su forma sí. Marcamos períodos
donde al menos dos de las tres señales están en su cuartil superior: son los mejores
candidatos a "cambio de agenda" porque no dependen de una sola métrica.

In [ ]:
def normalizar_01(s: pd.Series) -> pd.Series:
    return (s - s.min()) / (s.max() - s.min())

resumen = pd.DataFrame({
    "N_documentos": N_por_periodo,
    "pct_fuente_dominante": pct_fuente_dominante.round(3),
    "js_divergence": js_divergence.round(4),
    "cosine_distance": cosine_dist.round(4),
    "n_terminos_en_rafaga": n_rafagas_por_periodo,
})

js_norm = normalizar_01(js_divergence.fillna(0))
cos_norm = normalizar_01(cosine_dist.fillna(0))
burst_norm = normalizar_01(n_rafagas_por_periodo.astype(float))

umbral = 0.75
señales_altas = (
    (js_norm > js_norm.quantile(umbral)).astype(int)
    + (cos_norm > cos_norm.quantile(umbral)).astype(int)
    + (burst_norm > burst_norm.quantile(umbral)).astype(int)
)
candidatos_cambio = señales_altas[señales_altas >= 2].index.tolist()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(range(n_periodos), js_norm.values, label="JS divergence (norm.)", color="#444")
ax.plot(range(n_periodos), cos_norm.values, label="cosine distance (norm.)", color="crimson")
ax.plot(range(n_periodos), burst_norm.values, label="# términos en ráfaga (norm.)", color="steelblue")
for periodo in candidatos_cambio:
    ax.axvline(periodo_idx[periodo], color="black", alpha=0.15, lw=6)
ax.set_xticks(range(0, n_periodos, 4))
ax.set_xticklabels([periodos_orden[i] for i in range(0, n_periodos, 4)], rotation=90)
ax.legend()
ax.set_title("Las tres señales normalizadas — sombras verticales = ≥2 señales de acuerdo")
plt.tight_layout()
plt.show()

print(f"Períodos candidatos a cambio de agenda (≥2 señales de acuerdo): {candidatos_cambio}")

## 7. Guardar resumen y cerrar

In [ ]:
resumen.to_csv(RESULTS / "divergencia_lexica_por_periodo.csv")
print(f"guardado: {RESULTS / 'divergencia_lexica_por_periodo.csv'}")
resumen.loc[candidatos_cambio]

## Limitaciones e interpretación

- **Ningún pico se interpreta todavía contra la grilla histórica** (sección 7 y 23 de
  `CLAUDE.md`: descubrimiento primero). Esa comparación es la Etapa 7, después de que
  el modelado de tópicos confirme o no estos candidatos.
- **Los períodos sombreados en rojo (sección 3)** tienen N bajo o están dominados por
  una sola fuente — si un "candidato a cambio" cae ahí, no debe presentarse con la
  misma confianza que uno en un período bien cubierto. Esto es especialmente relevante
  para 1980–1989, donde el corpus es casi solo *Semana*.
- **Recomendación de robustez** (no implementada aquí, para no inflar este notebook):
  repetir las secciones 3-6 excluyendo 1980–1989 y comparar si los candidatos de cambio
  cambian — si los mismos períodos post-1990 siguen apareciendo, es evidencia de que no
  dependen de la cola dispersa del corpus.
- **Esto no reemplaza el modelado de tópicos** (Etapa 4): dice *cuándo* cambia el
  vocabulario agregado, no *qué* tema aparece o desaparece. Los períodos candidatos de
  la sección 6 son las ventanas donde vale la pena mirar con más detalle qué tópicos de
  LDA/NMF se mueven, en vez de revisar los ~90 períodos uno por uno.

### Estado tras la Entrega 2

Este experimento se hizo después de la Entrega 1 y antes de la Entrega 2, y es el punto de partida del enfoque final. Lo que se aprendió y cómo se corrigió:

- **La regla "≥ 2 de 3 señales en el cuartil superior" marca ~25 % de los periodos por construcción**, haya cambio o no. En `06-agenda-topicos-cambio.ipynb` se reemplaza por un modelo nulo (permutaciones estratificadas por fuente) que calibra qué es un cambio.
- **Resolución mixta (bienio hasta 1989, trimestre desde 1990)**: se reemplaza por resolución anual homogénea.
- **Fuente confundida con el tiempo**: el salto de 1989 a 1990 es la llegada de El Tiempo. La serie principal pasa a ser El Tiempo 1990–2011, con Semana y Dinero como contraste.
- **Robustez sin 1980–1989** (recomendada arriba): implementada en el 06.
- **Top‑5000 dominado por nombres propios**: el 05 mide el peso de las entidades y el 06 repite el análisis sin ellas.
- Los resultados de este notebook se conservan como línea base léxica; no se usan como evidencia de cambio.
